# Week 5 — Naive Bayes + class thresholding (Kaggle S6E4)

Imports.

In [16]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score, recall_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

Load Kaggle files from `week 5` (`train.csv` for modeling, `test.csv` reference only).

In [17]:
base_dir = Path('/Users/mkenne16/Documents/Advanced Machine Learning/week 5')
train_path = base_dir / 'train.csv'
test_path = base_dir / 'test.csv'

if not train_path.exists():
    raise FileNotFoundError(f'Missing required file: {train_path}')

print('Using train file:', train_path)
if test_path.exists():
    print('Found test file (not used for modeling):', test_path)

df = pd.read_csv(train_path)
print(df.shape)
df.head()

Using train file: /Users/mkenne16/Documents/Advanced Machine Learning/week 5/train.csv
Found test file (not used for modeling): /Users/mkenne16/Documents/Advanced Machine Learning/week 5/test.csv
(630000, 21)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


Pick target column and inspect class balance.

In [18]:
# Set this explicitly if your target has a different name.
# Common options: 'target', 'Target', 'irrigation', 'Irrigation'
target_candidates = ['target', 'Target', 'irrigation', 'Irrigation', 'label', 'Label', 'class', 'Class']
TARGET_COL = next((c for c in target_candidates if c in df.columns), df.columns[-1])
print('Target column:', TARGET_COL)

print('Class counts:')
print(df[TARGET_COL].value_counts(dropna=False))

Target column: Irrigation_Need
Class counts:
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64


Drop obvious ID column if present.

In [19]:
id_like = [c for c in df.columns if c.lower() in ['id', 'row_id']]
if id_like:
    print('Dropping ID-like columns:', id_like)
    df = df.drop(columns=id_like)

Dropping ID-like columns: ['id']


Train/test split from Kaggle train data only.

In [20]:
X = df.drop(columns=[TARGET_COL]).copy()
y = df[TARGET_COL].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

Train shape: (504000, 19)
Test shape: (126000, 19)


Build GaussianNB pipeline (impute + one-hot for categoricals).

In [21]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), cat_cols),
    ],
    remainder='drop'
)

nb_pipe = Pipeline([
    ('prep', preprocess),
    ('model', GaussianNB())
])

Fit model and generate predicted probabilities.

In [22]:
nb_pipe.fit(X_train, y_train)
probs = nb_pipe.predict_proba(X_test)
classes = nb_pipe.named_steps['model'].classes_

print('Classes in probability order:', classes)
print('Probabilities shape:', probs.shape)

Classes in probability order: ['High' 'Low' 'Medium']
Probabilities shape: (126000, 3)


Baseline predictions: default highest-probability class.

In [23]:
baseline_pred = classes[np.argmax(probs, axis=1)]

print('Baseline classification report:')
print(classification_report(y_test, baseline_pred, digits=4))

Baseline classification report:
              precision    recall  f1-score   support

        High     0.5271    0.7751    0.6275      4202
         Low     0.8753    0.7977    0.8347     73983
      Medium     0.6986    0.7655    0.7305     47815

    accuracy                         0.7847    126000
   macro avg     0.7003    0.7794    0.7309    126000
weighted avg     0.7966    0.7847    0.7883    126000



Choose focus class (rare class by default), metric, and threshold.

In [24]:
# Focus class: rare class in training set (you can override manually)
class_counts = y_train.value_counts()
focus_class = class_counts.idxmin()

# Metric choice (change if you prefer): 'f1', 'recall', or 'balanced_accuracy'
metric_choice = 'f1'

# Threshold for focus class probability (tune this value)
threshold = 0.35

focus_idx = np.where(classes == focus_class)[0][0]
print('Focus class:', focus_class)
print('Chosen metric:', metric_choice)
print('Threshold:', threshold)

Focus class: High
Chosen metric: f1
Threshold: 0.35


Thresholded predictions for focus class.

In [25]:
threshold_pred = baseline_pred.copy()
force_focus = probs[:, focus_idx] >= threshold
threshold_pred[force_focus] = focus_class

print('Thresholded classification report:')
print(classification_report(y_test, threshold_pred, digits=4))

Thresholded classification report:
              precision    recall  f1-score   support

        High     0.4388    0.8022    0.5673      4202
         Low     0.8753    0.7977    0.8347     73983
      Medium     0.6922    0.7367    0.7138     47815

    accuracy                         0.7747    126000
   macro avg     0.6688    0.7789    0.7053    126000
weighted avg     0.7912    0.7747    0.7799    126000



Compare baseline vs thresholded metric for the focus class.

In [26]:
def focus_metric(y_true, y_pred, cls, metric='f1'):
    y_true_bin = (y_true == cls).astype(int)
    y_pred_bin = (y_pred == cls).astype(int)

    if metric == 'f1':
        return f1_score(y_true_bin, y_pred_bin, zero_division=0)
    if metric == 'recall':
        return recall_score(y_true_bin, y_pred_bin, zero_division=0)
    if metric == 'balanced_accuracy':
        return balanced_accuracy_score(y_true_bin, y_pred_bin)
    raise ValueError('metric must be one of: f1, recall, balanced_accuracy')

baseline_metric = focus_metric(y_test, baseline_pred, focus_class, metric_choice)
threshold_metric = focus_metric(y_test, threshold_pred, focus_class, metric_choice)

print('Baseline', metric_choice, ':', round(baseline_metric, 4))
print('Thresholded', metric_choice, ':', round(threshold_metric, 4))
print('Change:', round(threshold_metric - baseline_metric, 4))

Baseline f1 : 0.6275
Thresholded f1 : 0.5673
Change: -0.0602


Optional: quick sweep to help pick threshold.

In [27]:
rows = []
for t in np.linspace(0.1, 0.9, 17):
    pred_t = baseline_pred.copy()
    pred_t[probs[:, focus_idx] >= t] = focus_class
    rows.append({
        'threshold': round(float(t), 2),
        'f1': focus_metric(y_test, pred_t, focus_class, 'f1'),
        'recall': focus_metric(y_test, pred_t, focus_class, 'recall'),
        'balanced_accuracy': focus_metric(y_test, pred_t, focus_class, 'balanced_accuracy'),
    })

threshold_table = pd.DataFrame(rows).sort_values('f1', ascending=False)
threshold_table.head(10)

,threshold,f1,recall,balanced_accuracy
8,0.50,0.627493,0.775107,0.875558
9,0.55,0.627493,0.775107,0.875558
15,0.85,0.627493,0.775107,0.875558
14,0.80,0.627493,0.775107,0.875558
13,0.75,0.627493,0.775107,0.875558
12,0.70,0.627493,0.775107,0.875558
11,0.65,0.627493,0.775107,0.875558
10,0.60,0.627493,0.775107,0.875558
16,0.90,0.627493,0.775107,0.875558
7,0.45,0.607758,0.784864,0.878668


Discussion (edit after running).

- I focused on the **High** irrigation class because it is the rarest class (about 21k out of 630k rows in the full train data).
- I first tried a **0.35** probability threshold for the High class and tracked **F1** for that class.
- With the default prediction rule (argmax / effectively 0.5 behavior), High-class F1 was **0.6275**. With threshold = 0.35, High-class F1 dropped to **0.5673** (**-0.0602**).
- The tradeoff was clear: recall for High increased (**0.7751 -> 0.8022**), but precision dropped a lot (**0.5271 -> 0.4388**), so F1 got worse. Overall accuracy also dropped (**0.7847 -> 0.7747**).
- The threshold sweep confirms this: the best F1 values were at **0.50+** (same as baseline behavior), so pushing the threshold lower did not help this metric on this split.
- Compared with my earlier tree-based models (XGBoost/Random Forest), Naive Bayes is fast and gives decent class probabilities, but it seems less robust to threshold changes on this multiclass task. For this notebook, I would keep the default decision rule (or tune for recall only if catching High cases is more important than precision).